In [1]:
import os
from pathlib import Path
root = Path.cwd()
while not (root / ".git").exists():
    root = root.parent
os.chdir(root)
print(os.getcwd())

c:\Users\Alive\Documents\eeg-brain-networks-dementia


In [2]:
%%writefile src/graphs.py
import numpy as np
import networkx as nx


def _prep(W):
    """Weighted graph with edge length = 1 / weight (strong link = short distance)."""
    G = nx.from_numpy_array(W)
    for u, v, d in G.edges(data=True):
        d["length"] = 1.0 / max(d["weight"], 1e-6)
    return G


def _path_and_eff(G):
    """Characteristic path length and global efficiency from shortest paths on 'length'."""
    dist = dict(nx.all_pairs_dijkstra_path_length(G, weight="length"))
    vals = [dist[i][j] for i in G for j in G if i != j]
    return float(np.mean(vals)), float(np.mean([1.0 / v for v in vals]))


def small_world(W, n_rand=100, seed=42):
    """sigma = (C / C_rand) / (L / L_rand); random graphs shuffle the edge weights."""
    rng = np.random.default_rng(seed)
    iu = np.triu_indices_from(W, k=1)
    w = W[iu]
    G = _prep(W)
    L, _ = _path_and_eff(G)
    C = nx.average_clustering(G, weight="weight")
    Cr, Lr = [], []
    for _ in range(n_rand):
        Wr = np.zeros_like(W)
        Wr[iu] = rng.permutation(w)
        Wr = Wr + Wr.T
        Gr = _prep(Wr)
        Lr.append(_path_and_eff(Gr)[0])
        Cr.append(nx.average_clustering(Gr, weight="weight"))
    return (C / np.mean(Cr)) / (L / np.mean(Lr))


def graph_metrics(W, seed=42, n_rand=100):
    """Global weighted graph measures for one 19 x 19 connectivity matrix."""
    G = _prep(W)
    L, E = _path_and_eff(G)
    C = nx.average_clustering(G, weight="weight")
    comms = nx.community.louvain_communities(G, weight="weight", seed=seed)
    Q = nx.community.modularity(G, comms, weight="weight")
    S = float(np.mean([d for _, d in G.degree(weight="weight")]))
    return {"strength": S, "clustering": C, "path_length": L, "efficiency": E,
            "modularity": Q, "small_world": small_world(W, n_rand=n_rand, seed=seed)}


def binary_metrics(W, keep=(0.1, 0.2, 0.3)):
    """Robustness check: clustering and efficiency on graphs keeping the strongest edges."""
    iu = np.triu_indices_from(W, k=1)
    out = {}
    for p in keep:
        thr = np.quantile(W[iu], 1 - p)
        A = (W >= thr).astype(int)
        np.fill_diagonal(A, 0)
        G = nx.from_numpy_array(A)
        out[f"bin{int(p * 100)}_clustering"] = nx.average_clustering(G)
        out[f"bin{int(p * 100)}_efficiency"] = nx.global_efficiency(G)
    return out

Writing src/graphs.py


In [3]:
print(os.listdir("src"))

['.gitkeep', 'connectivity.py', 'epoching.py', 'graphs.py', 'spectral.py', '__pycache__']


In [4]:
import sys, time
import numpy as np
sys.path.append("src")
import graphs

for sub in ["sub-001", "sub-037", "sub-066"]:
    W = np.load(f"data/connectivity/{sub}_wpli.npy")[2]      # band index 2 = alpha
    t = time.time()
    m = graphs.graph_metrics(W)
    print(sub, {k: round(v, 3) for k, v in m.items()}, f"{time.time() - t:.1f}s")
    print("   ", {k: round(v, 3) for k, v in graphs.binary_metrics(W).items()})

sub-001 {'strength': 3.389, 'clustering': 0.316, 'path_length': 5.701, 'efficiency': 0.2, 'modularity': 0.023, 'small_world': np.float64(0.969)} 2.6s
    {'bin10_clustering': 0.128, 'bin10_efficiency': 0.18, 'bin20_clustering': 0.241, 'bin20_efficiency': 0.385, 'bin30_clustering': 0.424, 'bin30_efficiency': 0.498}
sub-037 {'strength': 5.553, 'clustering': 0.447, 'path_length': 3.374, 'efficiency': 0.333, 'modularity': 0.004, 'small_world': np.float64(0.974)} 2.9s
    {'bin10_clustering': 0.092, 'bin10_efficiency': 0.219, 'bin20_clustering': 0.123, 'bin20_efficiency': 0.393, 'bin30_clustering': 0.439, 'bin30_efficiency': 0.497}
sub-066 {'strength': 3.769, 'clustering': 0.406, 'path_length': 5.245, 'efficiency': 0.216, 'modularity': 0.063, 'small_world': np.float64(0.946)} 3.1s
    {'bin10_clustering': 0.105, 'bin10_efficiency': 0.245, 'bin20_clustering': 0.475, 'bin20_efficiency': 0.413, 'bin30_clustering': 0.538, 'bin30_efficiency': 0.527}


In [2]:
import os
from pathlib import Path
root = Path.cwd()
while not (root / ".git").exists():
    root = root.parent
os.chdir(root)
print(os.getcwd())

c:\Users\Alive\Documents\eeg-brain-networks-dementia


In [3]:
print(os.listdir("src"))

['.gitkeep', 'connectivity.py', 'epoching.py', 'graphs.py', 'spectral.py', '__pycache__']


In [1]:
import os
from pathlib import Path
root = Path.cwd()
while not (root / ".git").exists():
    root = root.parent
os.chdir(root)
print(os.getcwd())

c:\Users\Alive\Documents\eeg-brain-networks-dementia


In [2]:
%%writefile src/graphs.py
import numpy as np
import networkx as nx


def _prep(W):
    """Weighted graph with edge length = 1 / weight (strong link = short distance)."""
    G = nx.from_numpy_array(W)
    for u, v, d in G.edges(data=True):
        d["length"] = 1.0 / max(d["weight"], 1e-6)
    return G


def _path_and_eff(G):
    """Characteristic path length and global efficiency from shortest paths on 'length'."""
    dist = dict(nx.all_pairs_dijkstra_path_length(G, weight="length"))
    vals = [dist[i][j] for i in G for j in G if i != j]
    return float(np.mean(vals)), float(np.mean([1.0 / v for v in vals]))


def small_world(W, n_rand=100, seed=42):
    """sigma = (C / C_rand) / (L / L_rand), plus the three ratios to weight-shuffled graphs."""
    rng = np.random.default_rng(seed)
    iu = np.triu_indices_from(W, k=1)
    w = W[iu]
    G = _prep(W)
    L, E = _path_and_eff(G)
    C = nx.average_clustering(G, weight="weight")
    Cr, Lr, Er = [], [], []
    for _ in range(n_rand):
        Wr = np.zeros_like(W)
        Wr[iu] = rng.permutation(w)
        Wr = Wr + Wr.T
        Gr = _prep(Wr)
        l, e = _path_and_eff(Gr)
        Lr.append(l)
        Er.append(e)
        Cr.append(nx.average_clustering(Gr, weight="weight"))
    c_norm, l_norm, e_norm = C / np.mean(Cr), L / np.mean(Lr), E / np.mean(Er)
    return {"small_world": float(c_norm / l_norm), "c_norm": float(c_norm),
            "l_norm": float(l_norm), "e_norm": float(e_norm)}


def graph_metrics(W, seed=42, n_rand=100):
    """Global weighted graph measures for one 19 x 19 connectivity matrix."""
    G = _prep(W)
    L, E = _path_and_eff(G)
    C = nx.average_clustering(G, weight="weight")
    comms = nx.community.louvain_communities(G, weight="weight", seed=seed)
    Q = nx.community.modularity(G, comms, weight="weight")
    S = float(np.mean([d for _, d in G.degree(weight="weight")]))
    out = {"strength": S, "clustering": float(C), "path_length": L, "efficiency": E,
           "modularity": float(Q)}
    out.update(small_world(W, n_rand=n_rand, seed=seed))
    return out


def binary_metrics(W, keep=(0.1, 0.2, 0.3)):
    """Robustness check: clustering and efficiency on graphs keeping the strongest edges."""
    iu = np.triu_indices_from(W, k=1)
    out = {}
    for p in keep:
        thr = np.quantile(W[iu], 1 - p)
        A = (W >= thr).astype(int)
        np.fill_diagonal(A, 0)
        G = nx.from_numpy_array(A)
        out[f"bin{int(p * 100)}_clustering"] = float(nx.average_clustering(G))
        out[f"bin{int(p * 100)}_efficiency"] = float(nx.global_efficiency(G))
    return out

Overwriting src/graphs.py


In [3]:
import sys, time
import numpy as np, pandas as pd
sys.path.append("src")
import graphs, connectivity

df = pd.read_csv("data/ds004504/participants.tsv", sep="\t")
cache = "data/graph_metrics_partial.csv"
done = pd.read_csv(cache) if os.path.exists(cache) else pd.DataFrame()
rows = done.to_dict("records")
done_subs = set(done["participant_id"]) if len(done) else set()

t0 = time.time()
for k, sub in enumerate(df["participant_id"], 1):
    f = f"data/connectivity/{sub}_wpli.npy"
    if sub in done_subs or not os.path.exists(f):
        continue
    Wall = np.load(f)
    for b, band in enumerate(connectivity.BAND_NAMES):
        W = Wall[b]
        r = dict(participant_id=sub, band=band)
        r.update(graphs.graph_metrics(W))
        r.update(graphs.binary_metrics(W))
        rows.append(r)
    pd.DataFrame(rows).to_csv(cache, index=False)
    print(f"[{k}/88] {sub} done, {(time.time() - t0) / 60:.1f} min elapsed")

gm = pd.DataFrame(rows).merge(df[["participant_id", "Group"]], on="participant_id")
gm.to_csv("results/tables/graph_metrics.csv", index=False)
print(gm.shape, "NaN cells:", int(gm.isna().sum().sum()))
print(gm.groupby("band")[["strength", "clustering", "path_length", "efficiency",
                          "modularity", "small_world"]].agg(["min", "max"]).round(2))

[1/88] sub-001 done, 0.2 min elapsed
[2/88] sub-002 done, 0.4 min elapsed
[3/88] sub-003 done, 0.7 min elapsed
[4/88] sub-004 done, 1.0 min elapsed
[5/88] sub-005 done, 1.3 min elapsed
[6/88] sub-006 done, 1.5 min elapsed
[7/88] sub-007 done, 1.7 min elapsed
[8/88] sub-008 done, 1.9 min elapsed
[9/88] sub-009 done, 2.2 min elapsed
[10/88] sub-010 done, 2.4 min elapsed
[11/88] sub-011 done, 2.5 min elapsed
[12/88] sub-012 done, 2.7 min elapsed
[13/88] sub-013 done, 2.9 min elapsed
[14/88] sub-014 done, 3.1 min elapsed
[15/88] sub-015 done, 3.3 min elapsed
[16/88] sub-016 done, 3.5 min elapsed
[17/88] sub-017 done, 3.7 min elapsed
[18/88] sub-018 done, 4.1 min elapsed
[19/88] sub-019 done, 4.4 min elapsed
[20/88] sub-020 done, 4.6 min elapsed
[21/88] sub-021 done, 4.8 min elapsed
[22/88] sub-022 done, 5.0 min elapsed
[23/88] sub-023 done, 5.2 min elapsed
[24/88] sub-024 done, 5.4 min elapsed
[25/88] sub-025 done, 5.7 min elapsed
[26/88] sub-026 done, 6.0 min elapsed
[27/88] sub-027 done,

In [5]:
import os
from pathlib import Path
root = Path.cwd()
while not (root / ".git").exists():
    root = root.parent
os.chdir(root)
print(os.getcwd())

import numpy as np, pandas as pd
from scipy.stats import kruskal, mannwhitneyu, spearmanr
from statsmodels.stats.multitest import multipletests

gm = pd.read_csv("results/tables/graph_metrics.csv")
spec = pd.read_csv("results/tables/spectral_features.csv")
df = pd.read_csv("data/ds004504/participants.tsv", sep="\t")

PRIMARY = ["strength", "clustering", "path_length", "efficiency", "modularity", "small_world"]
EXPLORATORY = ["c_norm", "l_norm", "e_norm"] + [c for c in gm.columns if c.startswith("bin")]
SPEC_COLS = ["relpow_delta", "relpow_theta", "relpow_alpha", "relpow_beta", "relpow_gamma", "theta_alpha_ratio"]
print(len(gm), len(spec), len(PRIMARY), len(EXPLORATORY))

c:\Users\Alive\Documents\eeg-brain-networks-dementia
435 87 6 9


In [6]:
PAIRS = [("A", "C"), ("F", "C"), ("A", "F")]

def run_tests(long, family):
    rows = []
    for (metric, band), d in long.groupby(["metric", "band"]):
        grp = {g: d.loc[d.Group == g, "value"].values for g in ["A", "F", "C"]}
        H, pk = kruskal(grp["A"], grp["F"], grp["C"])
        for g1, g2 in PAIRS:
            x, y = grp[g1], grp[g2]
            U, p = mannwhitneyu(x, y, alternative="two-sided")
            r = 2 * U / (len(x) * len(y)) - 1          # positive: first group higher
            rows.append(dict(family=family, metric=metric, band=band, pair=f"{g1}-{g2}",
                             n1=len(x), n2=len(y), median1=np.median(x), median2=np.median(y),
                             U=U, p=p, r_rb=r, kw_p=pk))
    res = pd.DataFrame(rows)
    res["p_fdr"] = multipletests(res.p, method="fdr_bh")[1]
    kw = res.drop_duplicates(["metric", "band"])[["metric", "band", "kw_p"]].copy()
    kw["kw_p_fdr"] = multipletests(kw.kw_p, method="fdr_bh")[1]
    return res.merge(kw[["metric", "band", "kw_p_fdr"]], on=["metric", "band"])

def melt_graph(cols):
    return gm.melt(id_vars=["participant_id", "band", "Group"], value_vars=cols,
                   var_name="metric", value_name="value")

res_primary = run_tests(melt_graph(PRIMARY), "primary_graph")
res_explor = run_tests(melt_graph(EXPLORATORY), "exploratory_graph")

sp = spec.melt(id_vars=["participant_id", "Group"], value_vars=SPEC_COLS,
               var_name="metric", value_name="value")
sp["band"] = "spectral"
res_spec = run_tests(sp, "spectral")

allres = pd.concat([res_primary, res_explor, res_spec]).sort_values(["family", "p_fdr"])
allres.to_csv("results/tables/group_stats.csv", index=False)

show = ["metric", "band", "pair", "median1", "median2", "r_rb", "p", "p_fdr"]
for fam, d in allres.groupby("family"):
    print(f"\n{fam}: {int((d.p_fdr < 0.05).sum())} of {len(d)} pairwise tests survive FDR (q = 0.05)")
    print(d[d.p_fdr < 0.05][show].round(4).to_string(index=False))


exploratory_graph: 0 of 135 pairwise tests survive FDR (q = 0.05)
Empty DataFrame
Columns: [metric, band, pair, median1, median2, r_rb, p, p_fdr]
Index: []

primary_graph: 1 of 90 pairwise tests survive FDR (q = 0.05)
    metric  band pair  median1  median2    r_rb   p  p_fdr
clustering theta  A-C   0.3911   0.4966 -0.6379 0.0  0.001

spectral: 5 of 18 pairwise tests survive FDR (q = 0.05)
           metric     band pair  median1  median2    r_rb      p  p_fdr
theta_alpha_ratio spectral  A-C   2.8183   1.4064  0.5785 0.0001 0.0012
     relpow_theta spectral  A-C   0.1026   0.0916  0.5000 0.0006 0.0053
     relpow_alpha spectral  A-C   0.0390   0.0653 -0.4693 0.0013 0.0075
theta_alpha_ratio spectral  F-C   2.1998   1.4064  0.5047 0.0023 0.0102
     relpow_alpha spectral  F-C   0.0413   0.0653 -0.4263 0.0100 0.0359


In [7]:
wide = gm.pivot(index="participant_id", columns="band", values=PRIMARY)
wide.columns = [f"{m}_{b}" for m, b in wide.columns]
wide = wide.join(spec.set_index("participant_id")[SPEC_COLS])
wide = wide.join(df.set_index("participant_id")[["Group", "MMSE", "Age", "Gender"]])
feats = [c for c in wide.columns if c not in ["Group", "MMSE", "Age", "Gender"]]

rows = []
for label, groups in [("AD", ["A"]), ("FTD", ["F"]), ("AD+FTD", ["A", "F"])]:
    s = wide[wide.Group.isin(groups)]
    for f in feats:
        rho, p = spearmanr(s[f], s["MMSE"])
        rows.append(dict(subset=label, feature=f, n=len(s), rho=rho, p=p))
mm = pd.DataFrame(rows)
mm["p_fdr"] = mm.groupby("subset")["p"].transform(lambda s: multipletests(s, method="fdr_bh")[1])
mm.sort_values(["subset", "p_fdr"]).to_csv("results/tables/mmse_correlations.csv", index=False)
print(mm[mm.p_fdr < 0.05].round(4).to_string(index=False))
print("Survivors per subset:", mm.assign(ok=mm.p_fdr < 0.05).groupby("subset").ok.sum().to_dict())

Empty DataFrame
Columns: [subset, feature, n, rho, p, p_fdr]
Index: []
Survivors per subset: {'AD': 0, 'AD+FTD': 0, 'FTD': 0}


In [8]:
import statsmodels.formula.api as smf

w = wide.copy()
w["rank_power"] = w["relpow_alpha"].rank()
print("Spearman, alpha strength vs alpha relative power, by group:")
for g in ["C", "F", "A"]:
    s = w[w.Group == g]
    print("  ", g, round(spearmanr(s["strength_alpha"], s["relpow_alpha"])[0], 2))

e_alpha = gm[gm.band == "alpha"].set_index("participant_id")["e_norm"].rename("e_norm_alpha")
w = w.join(e_alpha)
for target in ["strength_alpha", "e_norm_alpha"]:
    w["rank_y"] = w[target].rank()
    for adj in ["", " + rank_power"]:
        m = smf.ols(f"rank_y ~ C(Group, Treatment('C')){adj}", data=w).fit()
        print(f"\n{target}, model: group{adj}")
        print(m.summary().tables[1])

Spearman, alpha strength vs alpha relative power, by group:
   C 0.65
   F 0.82
   A 0.74

strength_alpha, model: group
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                        54.4828      4.534     12.016      0.000      45.466      63.499
C(Group, Treatment('C'))[T.A]   -15.4272      6.093     -2.532      0.013     -27.543      -3.312
C(Group, Treatment('C'))[T.F]   -16.2100      6.903     -2.348      0.021     -29.938      -2.482

strength_alpha, model: group + rank_power
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                         9.3627      5.067      1.848      0.068      -0.714      19.440
C(Group, Treatment('C'))[T.A]     0.7603      4.200  